# Functional or object-oriented?

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*

You know how to write functions and classes, and you have seen polymorphism, the main
reason to write classes, in [Inheritance](./classes_inheritance.ipynb). The block closes
with the question of **design**: what else classes solve, when they are the wrong answer,
and one problem solved in several styles.

**Objectives**

* know the problems that classes solve, and when a function is enough;
* compare the styles on the same problem: readability, testing, extension, speed;
* know which kind of change each style makes easy;
* choose, with reasons.

In [ ]:
import math
import random
from dataclasses import dataclass
import numpy as np

## 1. What classes solve

Polymorphism is one reason to write a class. There are two more, and they appear as soon
as a program grows. We start from a particle moving under a force, with functions alone:

In [ ]:
def step(x, v, m, force, dt):
    a = force / m
    return x + v * dt, v + a * dt

x, v, m = 0., 1., 2.
for _ in range(3):
    x, v = step(x, v, m, force=-1., dt=0.1)
print(f'x = {x:.3f}, v = {v:.3f}')

This works, and it is clear. For one particle and one afternoon, stop here: you do not
need a class.

### 1.1 Data that belongs together travels apart

`x`, `v` and `m` describe **one** particle, but the language does not know that.
Nothing stops us from mixing them up.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — A second particle</b></p>

The next cell moves a second particle. It runs without errors. Is the result right?

</div>

In [ ]:
x2, v2, m2 = 5., -1., 3.

x2, v2 = step(x2, v, m2, force=-1., dt=0.1)
print(f'x2 = {x2:.3f}, v2 = {v2:.3f}')

Look at the call: `v`, not `v2`. No error, no warning, a wrong result. With three
particles and three quantities each you carry nine loose variables, and one of these
mistakes becomes almost certain.

**A class binds the data that belongs together into one thing you cannot take apart by
accident.**

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        self.x, self.v, self.m = x, v, m

    def step(self, force, dt):
        self.v += (force / self.m) * dt
        self.x += self.v * dt

p = Particle(5., -1., 3.)
p.step(force=-1., dt=0.1)
print(f'x = {p.x:.3f}, v = {p.v:.3f}')

Now `step` cannot be called with *this* particle's position and *that* particle's
velocity.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Same physics?</b></p>

Compare the function `step` and the method `Particle.step`. For the same particle,
`x = 5, v = -1, m = 3`, one step with `force = -1` and `dt = 0.1`: do they give the same
`x`? Run the next cell. Which one is right? What happened?

</div>

In [ ]:
print(step(5., -1., 3., -1., 0.1))
q = Particle(5., -1., 3.)
q.step(-1., 0.1)
print(q.x, q.v)

<details>
<summary><b>Solution</b></summary>

Both are valid, but they are **different algorithms**. The function updates the position
with the *old* velocity (explicit Euler); the method updates the velocity first and moves
with the *new* one (semi-implicit, or symplectic, Euler). The second one is better for
oscillators: its energy error stays bounded.

The lesson is not about Euler: rewriting code into a class (a *refactor*) changed the
results, and nobody noticed. A refactor must keep the tests passing, so write the tests
**before** you refactor. (To make them agree, move `self.x += self.v * dt` before the
velocity update.)

</details>

### 1.2 Things that must always be true

A mass is positive. A direction vector has unit length. A probability lies in [0, 1].
These are **invariants**. With loose variables, every piece of code that touches the data
must remember to check them. With a class there is **one place**: the constructor, and a
property if the attribute can change later ([Writing classes](./classes.ipynb), section 7).

In [ ]:
class Particle:
    def __init__(self, x, v, m):
        if m <= 0:
            raise ValueError(f'mass must be positive, got {m}')
        self.x, self.v, self.m = x, v, m

try:
    Particle(0., 1., m=-2.)
except ValueError as e:
    print('caught:', e)

The object cannot be **created** invalid. Most long debugging sessions are spent finding
out *where* the data went wrong, not *what* went wrong.

### 1.3 What a class is

Three things bound together:

| | |
|:--|:--|
| **State** | the data that describes one object (`x`, `v`, `m`) |
| **Behaviour** | what you can do with it (`step`, `force`) |
| **A promise** | the *interface*: any object with a `force(x, v)` method can be used by `simulate` |

The promise is the part people forget, and the part that does the work.

And the four words people will throw at you:

| Word | What it actually means | Honest weight |
|:--|:--|:--|
| **Encapsulation** | keep the data and the code that maintains it together | useful |
| **Abstraction** | expose what it does, hide how | useful |
| **Polymorphism** | different objects, same interface | **the reason classes exist** |
| **Inheritance** | a class built on another | overused: prefer composition |

Textbooks give all four equal billing. They are not equal.

## 2. When *not* to write a class

Python is not Java: no rule says that code must live in a class. A class that keeps no
state is a function wearing a costume.

In [ ]:
# an anti-pattern: no state, one method
class EnergyCalculator:
    def compute(self, m):
        return m * 299792458. ** 2

print(EnergyCalculator().compute(1.))


# the same thing, better
def rest_energy(m):
    return m * 299792458. ** 2

print(rest_energy(1.))

**If it has no state and one method, it is a function. Write the function.** And if it
is data with names and no behaviour, a `@dataclass` ([Writing
classes](./classes.ipynb), section 8) is enough.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Class or function?</b></p>

Two classes from a real analysis code. What would you do with each one, and why?

```python
class Calibration:
    def __init__(self, slope, offset):
        self.slope, self.offset = slope, offset

    def apply(self, adc):
        return self.slope * adc + self.offset

class Peak:
    def __init__(self, mean, sigma, area):
        self.mean = mean
        self.sigma = sigma
        self.area = area
```

</div>

<details>
<summary><b>Solution</b></summary>

`Calibration` is a borderline case: two parameters and one method. A function with two
parameters (or `functools.partial`) does the same; a frozen dataclass is better if the
calibration is stored, printed and compared. The class is justified only if it gains more
methods (the inverse, the error). `Peak` is data with names and no behaviour: a
`@dataclass` gives the same in a few lines, plus `__repr__` and `__eq__` for free (and
`frozen=True` if a fitted peak should not change).

</details>

## 3. When to reach for a class: a checklist

Write a class when you can answer *yes* to at least one of these:

1. Do several pieces of data always travel together? (With no behaviour: a `dataclass`.)
2. Is there something that must always be true about them?
3. Will there be **more than one implementation** of the same idea, and do you want the
   rest of the code not to care which?
4. Does the thing have a lifetime: created, used many times, changed, closed?

If the answer to all four is *no*, write a function.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Class, function or dataclass?</b></p>

The code below is procedural. **Decide** what should become a class, what should stay a
function, and what should be a `dataclass`. Justify each choice with the checklist; do
not write the code yet.

```python
def read_spectrum(path): ...
def calibrate(channels, slope, offset): ...
def subtract_background(counts, bkg): ...
def fit_peak(energies, counts, e_min, e_max): ...
def plot(energies, counts, title): ...
```

</div>

<details>
<summary><b>Solution</b></summary>

One reasonable answer (there are others):

* a `Spectrum` **class** holding `channels`, `counts` and, after calibration, `energies`
  (data that travels together, item 1; counts never negative, item 2), with methods
  `calibrate` (returns a new `Spectrum` with `energies = slope * channels + offset`),
  `subtract_background`, `fit_peak` and `plot`;
* the calibration `(slope, offset)` and the fit result `(mean, sigma, area, errors)` as
  **dataclasses**: data with names and no behaviour;
* `read_spectrum(path)` stays a **function** (or a `Spectrum.from_file` classmethod) that
  returns a `Spectrum`.

</details>

## 4. One problem, three styles

A calorimeter gives, for each hit, a channel number and an ADC count. We must:

1. **calibrate**: energy (MeV) $= a \, \mathrm{ADC} + b$;
2. **select** the hits above a threshold;
3. compute the **total** and the **mean** energy of the selected hits.

In [ ]:
rng = random.Random(1)
raw = [(rng.randrange(64), rng.randrange(4096)) for _ in range(1000)]   # (channel, adc)
SLOPE, OFFSET, THRESHOLD = 0.05, 0.2, 50.                             # MeV/ADC, MeV, MeV
print(raw[:3])

### 4.1 Functional: pure functions on plain data

In [ ]:
def calibrate(hits, slope, offset):
    return [(ch, slope * adc + offset) for ch, adc in hits]

def select(hits, threshold):
    return [(ch, e) for ch, e in hits if e > threshold]

def total_and_mean(hits):
    energies = [e for _, e in hits]
    return sum(energies), sum(energies) / len(energies)

print(total_and_mean(select(calibrate(raw, SLOPE, OFFSET), THRESHOLD)))

# a test needs only an input and the expected output
assert total_and_mean([(0, 10.), (1, 30.)]) == (40., 20.)

### 4.2 Data classes and functions

The same functions, but the data have names: `hit.energy` instead of `hit[1]`.

In [ ]:
@dataclass(frozen=True)
class Hit:
    channel: int
    energy: float

def calibrate_hits(raw, slope, offset):
    return [Hit(ch, slope * adc + offset) for ch, adc in raw]

def select_hits(hits, threshold):
    return [h for h in hits if h.energy > threshold]

def summary(hits):
    total = sum(h.energy for h in hits)
    return total, total / len(hits)

print(summary(select_hits(calibrate_hits(raw, SLOPE, OFFSET), THRESHOLD)))

### 4.3 A class with methods

The data and the operations together, in an object:

In [ ]:
class Sample:
    """Calibrated calorimeter hits."""

    def __init__(self, channels, energies):
        self.channels = list(channels)
        self.energies = list(energies)

    @classmethod
    def from_raw(cls, raw, slope, offset):
        return cls([ch for ch, _ in raw], [slope * adc + offset for _, adc in raw])

    def select(self, threshold):
        """A new Sample with the hits above threshold."""
        kept = [(ch, e) for ch, e in zip(self.channels, self.energies) if e > threshold]
        return Sample([ch for ch, _ in kept], [e for _, e in kept])

    def total(self):
        return sum(self.energies)

    def mean(self):
        return self.total() / len(self.energies)

s = Sample.from_raw(raw, SLOPE, OFFSET).select(THRESHOLD)
print(s.total(), s.mean())

t = Sample([0, 1], [10., 30.])
assert (t.total(), t.mean()) == (40., 20.)

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Which one would you keep?</b></p>

Compare the three versions:

1. Which one is the easiest to **read**? To **test**?
2. Which one would you rather extend if a second detector arrives, with a **different
   calibration** (quadratic, per channel)?
3. And if instead you need a **new operation** on the hits (an energy-weighted mean
   channel, a histogram)?

</div>

<details>
<summary><b>Solution</b></summary>

1. All three are easy to test, because all the operations are **pure**: `Sample.select`
   returns a new object and never changes `self`. What makes code hard to test is hidden
   state, not classes. The data class version reads best (`h.energy`).
2. A new **calibration** is a new function (functional) or a new constructor or class
   (object-oriented): both are easy here, because calibration is separate from the
   analysis. The workshop of section 6 does exactly this.
3. A new **operation** (an energy-weighted mean channel) is one new function in the
   functional style, and touches nothing else; in the object-oriented style it is a new
   method in `Sample`, and in every class that offers the same interface.

The styles really differ when the code works with **several kinds** of things: a new kind
over fixed operations is one new class in the object-oriented style, and the code that
uses it does not change; in the functional style every function with an `if/elif` on the
kind must be edited. A new operation over fixed kinds is the opposite. This trade-off has
a name, the *expression problem*. Real codes mix both styles.

</details>

## 5. And with many hits?

All three versions create one Python object (a tuple, a `Hit`, a float) per hit. With
$10^7$ hits that is slow. The fourth style keeps the **structure** in an object and the
**numbers** in numpy arrays.

In [ ]:
class ArraySample:
    """Calibrated calorimeter hits, stored as numpy arrays."""

    def __init__(self, channels, energies):
        self.channels = np.asarray(channels)
        self.energies = np.asarray(energies, dtype=float)

    @classmethod
    def from_raw(cls, raw, slope, offset):
        channels, adc = np.asarray(raw).T
        return cls(channels, slope * adc + offset)

    def select(self, threshold):
        mask = self.energies > threshold
        return ArraySample(self.channels[mask], self.energies[mask])

    def total(self):
        return self.energies.sum()

    def mean(self):
        return self.energies.mean()

a = ArraySample.from_raw(raw, SLOPE, OFFSET).select(THRESHOLD)
print(a.total(), a.mean())

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — How much faster?</b></p>

With $10^6$ hits, **bet** on the speed-up of `ArraySample` over the functional version of
section 4.1: 2, 10 or 100 times?

</div>

In [ ]:
big = [(rng.randrange(64), rng.randrange(4096)) for _ in range(1_000_000)]
big_array = np.array(big)

%timeit total_and_mean(select(calibrate(big, SLOPE, OFFSET), THRESHOLD))
%timeit ArraySample(big_array[:, 0], SLOPE * big_array[:, 1] + OFFSET).select(THRESHOLD).mean()

<details>
<summary><b>Solution</b></summary>

Typically between 10 and 100 times, depending on the machine. But be fair: the array
version starts from an array; converting the list costs about as much as the functional
pipeline. Keep the data in arrays from the start.

Note that `ArraySample` is functional inside: `select` returns a **new** object and never
modifies `self`, and numpy expressions are `map` and `filter` over whole arrays.

So the right design for a physics code is usually **not** a million small objects, but a
*few* objects holding **arrays**: one `Event` with arrays of hits, one `Detector` with
arrays of channels. Objects for the structure, arrays for the numbers, pure operations.

</details>

## 6. Practice

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — A new requirement</b></p>

The calibration changes: each channel has its own slope, given by a list `slopes` of 64
values. In pairs: one person adapts the functional version (4.1), the other the class
(4.3 or 5). **First** write one test for your version (for example: the mean of the
selected energies is above the threshold), then change the calibration, and check that
both versions give the same total and mean.

How many lines did each one change? Where did the change go?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
slopes = [0.05 * (1 + 0.01 * (ch % 5)) for ch in range(64)]

# functional: a new calibration function; select and total_and_mean do not change
def calibrate_per_channel(hits, slopes, offset):
    return [(ch, slopes[ch] * adc + offset) for ch, adc in hits]

total, mean = total_and_mean(select(calibrate_per_channel(raw, slopes, OFFSET), THRESHOLD))
assert mean > THRESHOLD

# object-oriented (arrays): a new constructor function; select, total, mean unchanged
def from_raw_per_channel(raw, slopes, offset):
    channels, adc = np.asarray(raw).T
    return ArraySample(channels, np.asarray(slopes)[channels] * adc + offset)

a = from_raw_per_channel(raw, slopes, OFFSET).select(THRESHOLD)
assert a.mean() > THRESHOLD
assert math.isclose(a.mean(), mean) and math.isclose(a.total(), total)
```

In both cases the change stays in one place, because both designs separate calibration
from analysis. The design matters more than the paradigm.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Review this design</b></p>

A student wrote this. What would you change?

```python
class Analysis:
    def __init__(self, filename):
        self.data = open(filename).read().split()
        self.result = None

    def run(self):
        total = 0
        for i in range(len(self.data)):
            total = total + float(self.data[i]) * 0.05 + 0.2
        self.result = total
        print('total', self.result)
```

</div>

<details>
<summary><b>Solution</b></summary>

* The constructor reads a file and never closes it; reading belongs in a function (or a
  `classmethod`) with `with open(...)`.
* `run` mixes calibration, sum and printing, and returns nothing: it cannot be tested.
* `self.result` is state that exists only after `run`: an object that is half-built.
* `print` inside the computation: return the value, print at the edge.
* Magic numbers, `range(len(...))`.

A functional core with pure, testable pieces, and an object (or none) only for what must
be kept together:

```python
def read_adc(path):
    with open(path) as f:
        return [float(x) for x in f.read().split()]

def calibrate_adc(adc, slope=0.05, offset=0.2):
    return [slope * a + offset for a in adc]

def total_energy(path):
    return sum(calibrate_adc(read_adc(path)))
```

</details>

And if the hits do not fit in memory, the functional style becomes a lazy pipeline of
[generators](./about_iterators.ipynb), one hit at a time:

In [ ]:
def stream_hits(n, seed=2):
    r = random.Random(seed)
    for _ in range(n):
        yield r.randrange(64), r.randrange(4096)

energies = (SLOPE * adc + OFFSET for _, adc in stream_hits(10**6))
print(sum(e for e in energies if e > THRESHOLD))

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Interfaces you already use</b></p>

1. Find one interface (a method or a protocol) that several classes of `numpy`,
   `matplotlib` or the standard library share, and name three of those classes.
2. Argue **against** one object per event for a Monte Carlo generator that produces 10
   million events. What would you use instead?

</div>

<details>
<summary><b>Solution</b></summary>

1. For example, anything with `len()` and indexing (lists, tuples, arrays, strings), any
   iterable in a `for`, or matplotlib's `Axes.plot`, which accepts lists, tuples, arrays
   and pandas Series alike.
2. Ten million Python objects cost memory and time (section 5). Better: one object, an
   `EventSample`, holding numpy arrays of energies, positions…, with methods that act on
   all the events at once; or a generator that yields blocks of events.

</details>

***[+] Lookout.*** *Functional core, imperative shell*: keep the computations pure, and the reading, writing
and plotting at the edges. Other languages take other sides: Julia builds on *multiple
dispatch* (functions that choose their method by the types of all the arguments: one
answer to the expression problem), Haskell
is purely functional, and C++ mixes both, like Python.

## 7. Closing the block

* **Functions**: small, pure, tested. They are the unit of everything else.
* **Iterators and generators**: data that flows, one item at a time.
* **Classes**: when data and behaviour belong together, when invariants must hold, and
  when several implementations share one interface (polymorphism). A class with no state
  and one method is a function.
* **Tests** for all of it, written first when you can.
* For many numbers: **arrays**, inside a few objects, with pure operations.

<details>
<summary><b>[i] Cheat sheet — Which style?</b></summary>

| If… | then… |
|:--|:--|
| it computes a value from its inputs | a **pure function** |
| it is data with names, little behaviour | a `@dataclass` (`frozen=True`) |
| there is state and behaviour that belong together | a **class** |
| there will be several implementations of one idea | a class per implementation, one **interface** |
| you will add many new operations | functions over simple data tend to be easier |
| you will add many new kinds of things | classes with a common interface tend to be easier |
| there are millions of identical items | **numpy arrays**, inside a few objects |
| the data come as a stream, or do not fit in memory | **generators** and a pipeline |

**Write a class** if at least one is *yes*: several pieces of data always travel together;
something must always be true about them; there will be more than one implementation of
the same idea; the thing has a lifetime (created, used, changed, closed).

</details>

### Check yourself

1. Name three problems that classes solve.
2. Why is a pure function easier to test than an object with hidden state?
3. Which change is easy in the functional style, and which in the object-oriented one?
4. Why is "one object per hit" slow, and what is the alternative?
5. Give one case where you would write a class, and one where you would not.

<details>
<summary><b>Solutions</b></summary>

1. Data that belongs together travels apart; invariants with no place to live; the same
   operation on different things (`if/elif` chains on the kind).
2. Its result depends only on its arguments: no state to prepare, nothing else to check.
3. New operations in the functional style; new kinds of things in the object-oriented one.
4. Each object costs memory and a Python-level loop; keep the numbers in numpy arrays
   inside a few objects.
5. A class: a detector model with parameters, state and several methods, with several
   implementations. Not a class: a unit conversion (a function) or a fit result (a
   dataclass).

</details>

End of the block. Back to the [introduction](./python_intro.ipynb).